In [ ]:
from env import config, BASE_DIR
from sqlalchemy import create_engine, text

In [ ]:
DATABASE_URL = config("FLIGHT_PRICES_DATABASE_URL", cast=str, default=None)
DATABASE_URL is not None

In [ ]:
engine = create_engine(str(DATABASE_URL))

In [ ]:
DATASET_DIR = BASE_DIR / "datasets"
DATASET_DIR.exists()

In [ ]:
FLIGHT_PRICES_CSV = DATASET_DIR / "itineraries-min-50k.csv"
FLIGHT_PRICES_CSV.exists()

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv(str(FLIGHT_PRICES_CSV))
df.head(n=10)

In [ ]:
df.dtypes

In [ ]:
df.destinationAirport.value_counts()

In [ ]:
df["flightDate"] = pd.to_datetime(df["flightDate"])

In [ ]:
group_by_columns = [
    "flightDate",
    "startingAirport",
    "destinationAirport",
    "isBasicEconomy",
    "isRefundable",
    "isNonStop",
    "segmentsAirlineName",
]
target_column = "totalFare"
desired_columns = group_by_columns + [target_column]
sub_df = df.copy()[desired_columns]
sub_df = sub_df.groupby(group_by_columns)[target_column].mean().reset_index()
sub_df["startingAirport"] = sub_df["startingAirport"].apply(lambda x: f"{x}".strip())
sub_df["destinationAirport"] = sub_df["destinationAirport"].apply(
    lambda x: f"{x}".strip()
)
sub_df["totalFare"] = sub_df["totalFare"].round(0).apply(lambda x: int(x))
sub_df.dropna(inplace=True)
sub_df.head(n=40)

In [ ]:
sub_df.dtypes

In [ ]:
out_df = sub_df.copy()

In [ ]:
table_name = "flight_prices"
out_df.to_sql(table_name, engine, if_exists="replace", index=True)

In [ ]:
out_df.shape

In [ ]:
with engine.connect() as conn:
    results = conn.execute(text(f"DESCRIBE {table_name}"))
    print(results.all())